Part 3: Geographic Analysis

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
import math
import plotly.express as px
import plotly.graph_objects as go

In [4]:
dir_processed = Path('../data/processed')
dir_processed.mkdir(exist_ok=True)
dir_raw = Path('../data/raw')
dir_raw.mkdir(exist_ok=True)

In [2]:
"function resets index if new dataframes returns jumbled indexes"

def df_index(dataframe):
    rows = len(dataframe)

    dataframe.index = range(1, rows+1)
    
    return dataframe

In [5]:
dataset = pd.read_csv(dir_processed / 'dataset.csv')
dataset.index = dataset.index + 1

In [ ]:
dataset_dates = pd.read_csv(dir_processed / 'dataset_dates.csv')
dataset_dates

In [271]:
def color_map_grey(category):
    colors = [
        "#C5CDD5", 
        "#C4CCD0", 
        "#D0CBC2",
        "#CBCBCB", 
        "#B8B8B8",
        "#C8C1B6", 
        "#C1C1C1", 
        "#A9A9A9", 
        "#B6AE9E",
        "#B0BAC5", 
        "#BBC3CD", 
        "#ACACAC", 
        "#909EAE", 
        "#B5BEC9", 
        "#808080", 
        "#6082B6", 
        "#4F4F4F",
        "#A49A87",
        "#87A0A4",
        "#C8CCD5"
        ] 
    return {event: colors[i] for i, event in enumerate(category)}

In [149]:
"refactored return statement after Lecture W03D04"

def color_map_pop(selected_category, category_list):
    return {event: "#5C8DC5" if category_list[i] == selected_category else "#D3D3D3" for i, event in enumerate(category_list)}

In [231]:
"refactored return statement after Lecture W03D04"

def color_map_pop_multiple(selected_categories, category_list):
        return {admin: "#5C8DC5" if admin in selected_categories else "#D3D3D3" for i, admin in enumerate(category_list)}

In [84]:
dataset_geo_by_interval = (
    dataset
    .groupby(['interval', 'admin1'], as_index=False)['event_type']
    .size()
)

df_index(dataset_geo_by_interval)

dataset_geo = (
    dataset
    .groupby(['admin1'], as_index=False)['event_type']
    .size()
)

df_index(dataset_geo)

dataset_geo.head(30)

,admin1,size
1,Al Hasakeh,1926
2,Aleppo,2667
3,Ar Raqqa,1259
4,As Sweida,1065
5,Damascus,471
6,Dara,1305
7,Deir ez Zor,2389
8,Hama,744
9,Homs,1163
10,Idleb,420


In [42]:
geo_admin1 = list(dataset_geo['admin1'].unique())

In [ ]:
dataset_geo

In [64]:
govs = ['Al Hasakeh', 'Aleppo', 'Deir ez Zor']

In [73]:
bar_geo = px.bar(
    dataset_geo,
    x='admin1',
    y='size',
    color_discrete_map=color_map_grey(geo_admin1),
    labels = {'admin1': 'Governorates', 'event_count': 'Number of events'},
)

bar_geo.update_layout(
    title = 'Geographic distribution of conflict events since the fall of the Assad regime',
    title_font_size = 14,
    font_size=10,
    yaxis= dict(tickvals = [0, 400, 800, 1200, 1600, 2000, 2400, 2800])
)

bar_geo.update_traces(marker_color='rgb(211,211,211)')


In [74]:
bar_geo.write_html("../docs/assets/bar_governorates.html")

In [ ]:
dataset_geo_by_interval

In [199]:
df_sd = (
    dataset
    .query("event_type == 'Strategic developments'")
    .groupby(['interval', 'admin1', 'sub_event_type'], as_index=False)
    .agg(event_count=('sub_event_type', 'size'))
)
df_sd

,interval,admin1,sub_event_type,event_count
0,0,Al Hasakeh,Change to group/activity,30
1,0,Al Hasakeh,Disrupted weapons use,4
2,0,Al Hasakeh,Looting/property destruction,2
3,0,Al Hasakeh,Non-violent transfer of territory,1
4,0,Aleppo,Agreement,1
...,...,...,...,...
927,18,Rural Damascus,Disrupted weapons use,2
928,18,Rural Damascus,Other,2
929,18,Tartous,Arrests,2
930,18,Tartous,Change to group/activity,1


In [89]:
bar_sd_geo = px.bar(
    dataset_geo_by_interval, 
    x='interval',
    y='size',
    color='admin1',
    color_discrete_map=color_map_grey(geo_admin1),
)

bar_sd_geo.update_layout(
    title = 'Geographic distribution of conflict events since the fall of the Assad regime',
    title_font_size = 14,
    font_size=10
    # yaxis= dict(tickvals = [0, 400, 800, 1200, 1600, 2000, 2400, 2800])
)

In [ ]:
bubble_geo_events = px.scatter(
    dataset_geo_by_interval, 
    x="interval", 
    y="size",
    size="size", 
    color="admin1",
    hover_name="admin1", 
    size_max=20, 
    color_discrete_map=color_map_grey(geo_admin1), 
    labels={'interval':"Months elapsed since 08/12/2024", 'size':"Number of events"})

bubble_geo_events.update_xaxes(dtick=1)


In [98]:
bubble_geo_events.update_layout(
    title = 'Timeline of conflict events by governorates',
    legend_title_text = "Governorates",
    title_font_size = 14,
    font_size=10    
)

In [99]:
bubble_geo_events.write_html("../docs/assets/bubble_events_admin1_interval.html")

Sub-section: Line charts representing events occurrence per governorate since the regime collapse

In [147]:
df_sd = (
    dataset
    .query("event_type == 'Strategic developments'")
    .groupby(['interval', 'admin1'], as_index=False)['sub_event_type']
    .size()
)

df_sd = pd.merge(df_sd, dataset_dates, on='interval')

df_sd

line_sd = px.line(
    df_sd, 
    x="interval", 
    y="size", 
    color='admin1',
    color_discrete_map=color_map_grey(geo_admin1)
)

line_sd.update_xaxes(dtick=1)

line_sd

repeat the analysis but this time keep only the governorates with the highest number of occurrences

In [ ]:
df_sd_selected = (
    df_sd
    .query("admin1 in ['Aleppo', 'Quneitra', 'Al Hasakeh', 'Deir ez Zor']")
)
df_sd_selected

In [133]:
line_sd_selected_admin = px.line(
    df_sd_selected, 
    x="interval", 
    y="size", 
    color='admin1',
    color_discrete_map=color_map_grey(list(df_sd_selected['admin1'].unique()))
)
line_sd_selected

line_sd_selected_admin.update_layout(
    title = 'Timeline of strategic developments in most affected governorates',
    legend_title_text = "Governorates",
    title_font_size = 14,
    font_size=10    
)

line_sd_selected_admin.update_xaxes(dtick=1)

In [134]:
line_sd_selected_admin.write_html("../docs/assets/line_sd.html")

In [ ]:
df_sd_tt = (
    df_sd
    .query("sub_event_type == 'Non-violent transfer of territory'")
)
df_sd_tt

In [ ]:
df_sd_tt["interval"] = df_sd_tt["interval"].astype('string')

intervals = ['0', '1', '4', '6', '13', '14', '15']

type(df_sd_tt['interval'][0])

In [ ]:
color_map_pop('13', intervals)

In [218]:

bar_sd_tt = px.bar(
    df_sd_tt, 
    x='admin1',
    y='event_count',
    color='interval',
    color_discrete_map=color_map_grey(intervals),
    category_orders = {'interval': ['13', '14', '15', '0', '1', '4', '6']},
    labels={'admin1':'Governorates', 'event_count': 'Number of events'}
    )

bar_sd_tt

bar_sd_tt.update_layout(
    title = 'Non-violent transfer of territory by governorate and interval',
    title_font_size = 14,
    font_size=10
    # yaxis= dict(tickvals = [0, 400, 800, 1200, 1600, 2000, 2400, 2800])
)

In [219]:
bar_sd_tt.write_html("../docs/assets/bar_non-violent_transfer_of_territory.html")

Geographic analysis of explosions/remote violence

In [ ]:
df_erv = (
    dataset
    .query("event_type == 'Explosions/Remote violence'")
    .groupby(['interval', 'admin1'], as_index=False)['sub_event_type']
    .size()
)

df_erv.head(30)

In [155]:
line_erv = px.line(
    df_erv, 
    x="interval", 
    y="size", 
    color='admin1',
    color_discrete_map=color_map_pop('Aleppo', geo_admin1),
    labels={"interval":"Months elapsed since 08/12/2024", "size":"Number of events"}
)

line_erv.update_layout(
    title = 'Timeline of explosions/remote violence across all governorates, Aleppo most affected in the immediate aftermath',
    legend_title_text = "Governorates",
    title_font_size = 14,
    font_size=10    
)

line_erv.update_xaxes(dtick=1)

In [156]:
line_erv.write_html("../docs/assets/line_erv.html")

Geographic analysis of violence against civilians

In [ ]:
df_vac = (
    dataset
    .query("event_type == 'Violence against civilians'")
    .groupby(['interval', 'admin1'], as_index=False)['sub_event_type']
    .size()
)

df_vac.head(30)

In [235]:
line_vac = px.line(
    df_vac, 
    x="interval", 
    y="size", 
    color='admin1',
    color_discrete_map=color_map_pop_multiple(['Al Hasakeh', 'Homs', 'Deir ez Zor', 'Aleppo'], geo_admin1),
    labels={"interval":"Months elapsed since 08/12/2024", "size":"Number of events"}
)

line_vac.update_layout(
    title = 'Timeline of violence against civilians across all governorates',
    legend_title_text = "Governorates",
    title_font_size = 14,
    font_size=10    
)

line_vac.update_xaxes(dtick=1)

In [236]:
line_vac.write_html("../docs/assets/line_vac.html")

In [266]:
# df_vac_afd = (
#     dataset
#     .query("sub_event_type == 'Abduction/forced disappearance'")
#     .groupby(['interval', 'admin1'], as_index=False)['sub_event_type']
#     .size()
# )
df_vac_afd[df_vac_afd['interval'] == '18']

# df_vac_afd["interval"] = df_vac_afd["interval"].astype('string')

# intervals_vac = list(df_vac_afd['interval'].unique())

# type(df_vac_afd['interval'][0])

,interval,admin1,size
152,18,Al Hasakeh,3
153,18,Ar Raqqa,2
154,18,Dara,1
155,18,Deir ez Zor,1
156,18,Quneitra,5


In [269]:

bar_vac_afd = px.bar(
    df_vac_afd, 
    x='admin1',
    y='size',
    color='interval',
    color_discrete_map=color_map_grey(intervals),
    labels={'admin1':'Governorates', 'size': 'Number of events'}
    )

bar_vac_afd

bar_vac_afd.update_layout(
    title = 'Abductions/forced disappearances by governorate and interval',
    title_font_size = 14,
    font_size=10
    # yaxis= dict(tickvals = [0, 400, 800, 1200, 1600, 2000, 2400, 2800])
)

IndexError: list index out of range

In [ ]:

bubble_vac_afd = px.scatter(
    df_vac_afd, 
    x='admin1',
    y='size',
    size='size',
    color='interval',
    hover_name="interval",
    size_max=20, 
    color_discrete_map=color_map_pop_multiple(['0', '1', '13'], intervals),
    labels={'admin1':'Governorates', 'size': 'Number of events'}
    )

bubble_vac_afd

bubble_vac_afd.update_layout(
    title = 'Abductions/forced disappearances by governorate and interval',
    title_font_size = 14,
    font_size=8
    # yaxis= dict(tickvals = [0, 400, 800, 1200, 1600, 2000, 2400, 2800])
)

In [ ]:
intervals = list(df_vac_afd['interval'].unique())
intervals

In [277]:
bubble_vac_afd.write_html("../docs/assets/bubble_vac_afd.html")